# Student Academic Risk: ES, LR, and SVM

This notebook compares three systems for binary student academic risk prediction.

| Abbreviation | Meaning | Role |
|---|---|---|
| **Common:** | Shared preprocessing | Used by all systems |
| **DT:** | Decision Tree | Generates IF–THEN rules |
| **KB:** | Knowledge Base | Stores generated rules |
| **ES:** | Expert System | Applies KB rules using Experta |
| **LR:** | Logistic Regression | Machine Learning baseline |
| **SVM:** | Support Vector Machine | Machine Learning baseline |

`G3` is converted into the binary target:

- `1 = At Risk` if `G3 < 10`
- `0 = Not At Risk` if `G3 >= 10`

## Common: Setup

Import libraries and define shared constants.


In [1]:
# Standard library imports
import json
import sys
import subprocess
import collections
import collections.abc
from pathlib import Path

# Data handling imports
import numpy as np
import pandas as pd

# Preprocessing and splitting imports
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# DT rule generation
from sklearn.tree import DecisionTreeClassifier, _tree

# ML baselines
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Fixed random seed for reproducibility
RANDOM_STATE = 42

# Display settings for clearer notebook outputs
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 140)


## Common: Load Dataset

Load the student dataset.


In [2]:
# Default dataset file.
# Change this to Path("student-por.csv") to use the Portuguese dataset.
DATA_PATH = Path("student-mat.csv")

# If the file is not beside the notebook, check the data/ folder.
if not DATA_PATH.exists():
    alternative_path = Path("data") / DATA_PATH.name

    # Use the data/ version if it exists.
    if alternative_path.exists():
        DATA_PATH = alternative_path

# Stop execution with a clear message if the dataset is missing.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset file not found. Place student-mat.csv beside this notebook or inside a data/ folder."
    )

# The UCI student dataset uses semicolon-separated values.
df = pd.read_csv(DATA_PATH, sep=";")

# Print a quick loading summary.
print(f"Loaded file: {DATA_PATH}")
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

# Show the first rows to confirm successful loading.
df.head()


Loaded file: data\student-mat.csv
Dataset shape: 395 rows, 33 columns


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4,6,10,10


## Common: Create Target and Features

Create the binary target from `G3`.

In [3]:
# These columns are required for this experiment.
required_columns = ["G3"]

# Check whether any required grade column is missing.
missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

# Raise an explicit error if the dataset does not match the expected structure.
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Copy the dataframe so the original loaded data remains unchanged.
data = df.copy()

# Convert G3 into a binary target.
# G3 < 10 means the student failed, so the student is considered At Risk.
# G3 >= 10 means the student passed, so the student is considered Not At Risk.
data["at_risk"] = (data["G3"] < 10).astype(int)

# X_raw contains all model inputs.
# G3 is removed because it was used to create the target.
# at_risk is removed because it is the target itself.
X_raw = data.drop(columns=["G3", "at_risk"])

# y contains the binary target labels.
y = data["at_risk"]

# Human-readable class names used in reports.
class_names = ["Not At Risk", "At Risk"]

print(f"Input shape: {X_raw.shape}")
print("G1 and G2 are included.")
print("G3 is removed from inputs.")


Input shape: (395, 32)
G1 and G2 are included.
G3 is removed from inputs.


## Common: Check Class Balance

Display the target distribution.


In [4]:
# Convert numeric target values to readable labels before counting.
target_distribution = (
    y.map({0: "Not At Risk", 1: "At Risk"})
    .value_counts()
    .to_frame("count")
)

# Add percentage of the full dataset for each class.
target_distribution["percent"] = (
    target_distribution["count"] / len(y) * 100
).round(2)

# Display class balance.
display(target_distribution)


,count,percent
at_risk,,
Not At Risk,265,67.09
At Risk,130,32.91


## Common: Identify Feature Types

Separate input columns into categorical and numeric features.


In [5]:
# Categorical features contain text/object values.
# These need one-hot encoding before any model can use them.
categorical_features = X_raw.select_dtypes(include="object").columns.tolist()

# Numeric features are already numeric.
numeric_features = X_raw.select_dtypes(exclude="object").columns.tolist()

# Print feature groups for review.
print("Categorical Features:")
print(", ".join(categorical_features))

print("\nNumeric Features:")
print(", ".join(numeric_features))


Categorical Features:
school, sex, address, famsize, Pstatus, Mjob, Fjob, reason, guardian, schoolsup, famsup, paid, activities, nursery, higher, internet, romantic

Numeric Features:
age, Medu, Fedu, traveltime, studytime, failures, famrel, freetime, goout, Dalc, Walc, health, absences, G1, G2


## Common: Split Data

Use the same train/test split for ES, LR, and SVM.


In [6]:
# Create one shared split for all systems.
# test_size=0.2 means 80% training and 20% testing.
# stratify=y keeps the class distribution similar in train and test sets.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Summarize train/test sizes and class proportions.
split_summary = pd.DataFrame({
    "set": ["train", "test"],
    "rows": [len(X_train_raw), len(X_test_raw)],
    "at_risk_percent": [
        round(y_train.mean() * 100, 2),
        round(y_test.mean() * 100, 2)
    ]
})

display(split_summary)


,set,rows,at_risk_percent
0,train,316,32.91
1,test,79,32.91


## Common: Build Shared Preprocessor

Create one shared preprocessing pipeline for ES, LR, and SVM.

Categorical features are one-hot encoded. Numeric features pass through unchanged.


In [7]:
# OneHotEncoder handles categorical columns whose values are names, not quantities.
# Example: Mjob has values such as teacher, health, services, at_home, and other.
# Encoding these as 0, 1, 2, 3, 4 would incorrectly imply a numeric order between them.
# One-hot encoding avoids this by creating one 0/1 column per category.
# A value of 1 means the student belongs to that category, and 0 means they do not.
try:
    shared_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    shared_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

# Common preprocessor used by ES, LR, and SVM.
common_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", shared_encoder, categorical_features),
        ("numeric", "passthrough", numeric_features)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)


## Common: Apply Shared Preprocessing

Fit the shared preprocessor on the training set and apply it to both training and test sets.


In [8]:
# Fit the preprocessor only on training data.
X_train_processed_array = common_preprocessor.fit_transform(X_train_raw)

# Apply the already-fitted preprocessing to the test set.
X_test_processed_array = common_preprocessor.transform(X_test_raw)

# Get the column names after one-hot encoding.
# Because one-hot encoding creates new columns for each category, the original column names are transformed.
processed_feature_names = common_preprocessor.get_feature_names_out().tolist()


## Common: Build Processed Dataframes

Store the processed arrays as dataframes.

These same processed dataframes are used by ES, LR, and SVM.


In [9]:
# Convert processed training array to dataframe.
X_train_processed = pd.DataFrame(
    X_train_processed_array,
    columns=processed_feature_names,
    index=X_train_raw.index
)

# Convert processed test array to dataframe.
X_test_processed = pd.DataFrame(
    X_test_processed_array,
    columns=processed_feature_names,
    index=X_test_raw.index
)

# The number of rows should stay the same.
# The number of columns usually increases because of one-hot encoding.
print(f"Raw Training Shape: {X_train_raw.shape}")
print(f"Processed Training Shape: {X_train_processed.shape}")

X_train_processed.head()


Raw Training Shape: (316, 32)
Processed Training Shape: (316, 58)


,school_GP,school_MS,sex_F,sex_M,address_R,address_U,famsize_GT3,famsize_LE3,Pstatus_A,Pstatus_T,Mjob_at_home,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_at_home,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_course,reason_home,reason_other,reason_reputation,guardian_father,guardian_mother,guardian_other,schoolsup_no,schoolsup_yes,famsup_no,famsup_yes,paid_no,paid_yes,activities_no,activities_yes,nursery_no,nursery_yes,higher_no,higher_yes,internet_no,internet_yes,romantic_no,romantic_yes,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2
101,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,16.0,4.0,4.0,1.0,3.0,0.0,4.0,4.0,3.0,1.0,1.0,4.0,0.0,16.0,17.0
63,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,16.0,4.0,3.0,1.0,3.0,0.0,3.0,4.0,4.0,2.0,4.0,4.0,2.0,10.0,9.0
245,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,16.0,2.0,1.0,3.0,1.0,0.0,4.0,3.0,3.0,1.0,1.0,4.0,6.0,18.0,18.0
112,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,16.0,2.0,2.0,1.0,2.0,1.0,3.0,1.0,2.0,1.0,1.0,5.0,6.0,10.0,13.0
8,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,15.0,3.0,2.0,1.0,2.0,0.0,4.0,2.0,2.0,1.0,1.0,1.0,0.0,16.0,18.0


## DT: Train Rule Generator

Train a small entropy-based DT.

The DT is used only to generate rules for the ES.


In [10]:
# Entropy makes the DT choose splits based on information gain.
# max_depth controls rule length.
# min_samples_split and min_samples_leaf prevent overly specific rules.
# class_weight="balanced" gives more training weight to the smaller class.
# This helps the model avoid ignoring At Risk students when the dataset has more Not At Risk examples.
dt_rule_generator = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

# Train the DT rule generator on the shared processed training data.
dt_rule_generator.fit(X_train_processed, y_train)

# Each leaf becomes one generated rule.
print(f"DT Depth: {dt_rule_generator.get_depth()}")
print(f"Generated Rules: {dt_rule_generator.get_n_leaves()}")


DT Depth: 4
Generated Rules: 9


## KB: Map Encoded Features

Map one-hot encoded feature names back to their original categorical meaning.


In [11]:
# The shared preprocessor contains multiple transformations.
# We named the OneHotEncoder part "categorical" when we created the ColumnTransformer.
# This line retrieves that fitted OneHotEncoder.
fitted_encoder = common_preprocessor.named_transformers_["categorical"]

# After fitting, the encoder knows the exact categories found in each categorical column.
# Example:
# school    -> GP, MS
# schoolsup -> no, yes
# Mjob      -> at_home, health, other, services, teacher
#
# We need this information because the DT rules use encoded feature names like:
# school_GP, schoolsup_yes, Mjob_teacher
#
# These names are useful for the model, but not very readable in the final KB.
# This dictionary maps each encoded feature name back to its original meaning.
one_hot_feature_map = {}

# Loop over each original categorical column and the categories learned for that column.
# zip(...) pairs each column name with its corresponding category list.
for original_column, categories in zip(categorical_features, fitted_encoder.categories_):

    # Loop over every category inside the current categorical column.
    for category in categories:

        # Rebuild the encoded feature name produced by OneHotEncoder.
        # Example:
        # original_column = "schoolsup", category = "yes"
        # encoded_name = "schoolsup_yes"
        encoded_name = f"{original_column}_{category}"

        # Store the meaning of this encoded feature.
        # This will later let us convert a DT condition like:
        # schoolsup_yes > 0.5
        # into readable text:
        # schoolsup is yes
        one_hot_feature_map[encoded_name] = {
            "original_column": original_column,
            "category": category
        }

## KB: Format Conditions

Define a helper function for converting DT conditions into readable text.


In [12]:
def readable_condition(feature, operator, threshold):
    """Convert a DT condition into readable rule text."""

    # One-hot encoded categorical columns usually split at threshold 0.5.
    # Example:
    # schoolsup_yes > 0.5 means schoolsup is yes.
    # schoolsup_yes <= 0.5 means schoolsup is not yes.
    if feature in one_hot_feature_map and abs(threshold - 0.5) < 1e-8:
        original_column = one_hot_feature_map[feature]["original_column"]
        category = one_hot_feature_map[feature]["category"]

        if operator == ">":
            return f"{original_column} is {category}"

        return f"{original_column} is not {category}"

    # Numeric feature split.
    # Round the threshold text to keep rules readable.
    threshold_text = f"{threshold:.3f}".rstrip("0").rstrip(".")
    return f"{feature} {operator} {threshold_text}"


## KB: Define Rule Extraction Helpers

Create helper functions that convert DT leaf paths into IF–THEN rules.


In [13]:
def create_rule_from_leaf(tree_, node_id, conditions, class_names):
    """Create one rule from one DT leaf node."""

    # Class distribution at the leaf.
    # Example: counts for [Not At Risk, At Risk].
    class_counts = tree_.value[node_id][0]

    # The predicted class is the class with the largest count at this leaf.
    predicted_class_code = int(np.argmax(class_counts))

    # Confidence is the proportion of the predicted class at this leaf.
    total_weighted_samples = float(class_counts.sum())
    confidence = 0.0

    if total_weighted_samples > 0:
        confidence = float(class_counts[predicted_class_code] / total_weighted_samples)

    # Store all information needed for inference and reporting.
    return {
        # Raw list of rule conditions.
        # Each condition stores the feature, operator, threshold, and readable text.
        "conditions": conditions,

        # Human-readable version of the full rule path.
        # Conditions inside one DT path are joined with AND.
        "conditions_text": " AND ".join(condition["text"] for condition in conditions),

        # Numeric predicted class from the leaf.
        # 0 = Not At Risk, 1 = At Risk.
        "prediction_code": predicted_class_code,

        # Readable class label using the numeric prediction.
        "prediction_label": class_names[predicted_class_code],

        # Rule confidence based on the class distribution at this leaf.
        "confidence": round(confidence, 4),

        # Number of training samples that reached this leaf.
        # This helps show how general or specific the rule is.
        "training_samples": int(tree_.n_node_samples[node_id])
    }


def traverse_tree(tree_, node_id, conditions, feature_names, class_names, rules):
    """Traverse the DT and collect one rule per leaf."""

    # The feature used for splitting at the current node.
    feature_index = tree_.feature[node_id]

    # If this is not a leaf, continue traversing both branches.
    if feature_index != _tree.TREE_UNDEFINED:
        feature = feature_names[feature_index]
        threshold = float(tree_.threshold[node_id])

        # Left branch condition: feature <= threshold.
        left_condition = {
            "feature": feature,
            "operator": "<=",
            "threshold": threshold,
            "text": readable_condition(feature, "<=", threshold)
        }

        # Right branch condition: feature > threshold.
        right_condition = {
            "feature": feature,
            "operator": ">",
            "threshold": threshold,
            "text": readable_condition(feature, ">", threshold)
        }

        # Traverse the left child and add the left condition to the path.
        traverse_tree(
            tree_,
            tree_.children_left[node_id],
            conditions + [left_condition],
            feature_names,
            class_names,
            rules
        )

        # Traverse the right child and add the right condition to the path.
        traverse_tree(
            tree_,
            tree_.children_right[node_id],
            conditions + [right_condition],
            feature_names,
            class_names,
            rules
        )

        return

    # If this is a leaf, convert the current path into one rule.
    rules.append(
        create_rule_from_leaf(
            tree_=tree_,
            node_id=node_id,
            conditions=conditions,
            class_names=class_names
        )
    )


def extract_rules_from_tree(tree_model, feature_names, class_names):
    """Extract all DT root-to-leaf paths as KB rules."""

    # This list will contain all extracted rules.
    rules = []

    # Start traversal from node 0, which is the root of the tree.
    traverse_tree(
        tree_=tree_model.tree_,
        node_id=0,
        conditions=[],
        feature_names=feature_names,
        class_names=class_names,
        rules=rules
    )

    # Add stable rule IDs after extraction.
    for rule_id, rule in enumerate(rules, start=1):
        rule["rule_id"] = rule_id

    return rules


## KB: Generate Knowledge Base

Extract rules from the DT and display them as the ES knowledge base.


In [14]:
# Extract generated rules from the trained DT.
knowledge_base = extract_rules_from_tree(
    tree_model=dt_rule_generator,
    feature_names=processed_feature_names,
    class_names=class_names
)

# Convert the KB to a readable dataframe.
kb_table = pd.DataFrame([
    {
        "rule_id": rule["rule_id"],
        "conditions": rule["conditions_text"],
        "prediction": rule["prediction_label"],
        "confidence": rule["confidence"],
        "training_samples": rule["training_samples"]
    }
    for rule in knowledge_base
])

display(kb_table)


,rule_id,conditions,prediction,confidence,training_samples
0,1,G2 <= 9.5 AND G2 <= 7.5,At Risk,1.0000,51
1,2,G2 <= 9.5 AND G2 > 7.5 AND Fedu <= 1.5 AND sex is not F,At Risk,0.8717,13
2,3,G2 <= 9.5 AND G2 > 7.5 AND Fedu <= 1.5 AND sex is F,Not At Risk,0.5954,12
3,4,G2 <= 9.5 AND G2 > 7.5 AND Fedu > 1.5 AND famrel <= 4.5,At Risk,0.9532,33
4,5,G2 <= 9.5 AND G2 > 7.5 AND Fedu > 1.5 AND famrel > 4.5,At Risk,0.7182,9
5,6,G2 > 9.5 AND G1 <= 10.5 AND age <= 17.5 AND famsize is not LE3,Not At Risk,1.0000,28
6,7,G2 > 9.5 AND G1 <= 10.5 AND age <= 17.5 AND famsize is LE3,Not At Risk,0.6319,9
7,8,G2 > 9.5 AND G1 <= 10.5 AND age > 17.5,At Risk,0.7536,5
8,9,G2 > 9.5 AND G1 > 10.5,Not At Risk,1.0000,156


## KB: Export Knowledge Base

Save generated rules for reporting and reuse.


In [15]:
# Save readable KB table for the report.
kb_table.to_csv("generated_expert_system_rules.csv", index=False)

# Save full KB structure for reuse in code.
with open("generated_expert_system_knowledge_base.json", "w", encoding="utf-8") as file:
    json.dump(knowledge_base, file, indent=2)

print("Saved generated_expert_system_rules.csv")
print("Saved generated_expert_system_knowledge_base.json")


Saved generated_expert_system_rules.csv
Saved generated_expert_system_knowledge_base.json


## ES: Import Experta

Import Experta for rule-based inference.


In [16]:
# Experta may depend on old collections names.
# This patch maps old names to their current locations.
for name in ["Mapping", "MutableMapping", "Sequence"]:
    if not hasattr(collections, name):
        setattr(collections, name, getattr(collections.abc, name))

# Try importing Experta.
# If missing, install it and import again.
try:
    from experta import KnowledgeEngine, Fact, Rule, AS
except ModuleNotFoundError:
    print("Experta is not installed. Installing experta...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "experta"])
    from experta import KnowledgeEngine, Fact, Rule, AS

print("Experta imported successfully.")


Experta imported successfully.


## ES: Define Inference Helpers

Define how the ES checks whether a KB rule matches a student.


In [17]:
class Student(Fact):
    """One processed student record."""
    pass


def condition_matches(value, operator, threshold):
    """Evaluate one extracted KB condition for the current student."""

    # The DT learned these conditions, but after extraction they are stored as plain data.
    # The ES needs to interpret the stored operator and threshold during inference.
    if operator == "<=":
        return value <= threshold

    if operator == ">":
        return value > threshold

    raise ValueError(f"Unsupported operator: {operator}")


def rule_matches_student(student_features, rule):
    """Check whether all conditions in one rule match a student."""

    # Conditions inside one rule are AND conditions.
    for condition in rule["conditions"]:
        feature = condition["feature"]
        operator = condition["operator"]
        threshold = condition["threshold"]
        
        # Get the current student's value for the feature used in this rule condition.
        # For numeric features, this may be a value like G2 = 8 or absences = 12.
        # For one-hot encoded categorical features, this will be 0 or 1.
        # The default value 0 means "category not present" if an encoded feature is missing.
        value = float(student_features.get(feature, 0))

        # If any condition is false, this rule does not match.
        if not condition_matches(value, operator, threshold):
            return False

    # All conditions are true, so this rule matches.
    return True


## ES: Define Engine

Define the Experta engine that applies KB rules.


In [18]:
class AcademicRiskExpertSystem(KnowledgeEngine):
    """Experta engine for academic risk prediction."""

    def __init__(self, rules):
        super().__init__()

        # Store generated KB rules.
        self.rules = rules

        # Store the output after inference.
        self.result = None

    # This Experta rule fires when a Student fact is added to the engine.
    # AS.student captures the matched Student fact and passes it to the function.
    @Rule(AS.student << Student())
    def infer_academic_risk(self, student):
        """Infer risk class for one student."""

        # Convert Experta fact into a normal dictionary.
        student_features = {
            key: value
            for key, value in dict(student).items()
            if key in processed_feature_names
        }

        # Find all KB rules that match this student.
        matching_rules = [
            rule for rule in self.rules
            if rule_matches_student(student_features, rule)
        ]

        # Safety fallback.
        # A complete DT-derived KB should normally match exactly one rule.
        if not matching_rules:
            self.result = {
                "prediction_code": None,
                "prediction_label": "Unknown",
                "rule_id": None,
                "confidence": 0.0,
                "explanation": "No rule matched this student."
            }
            return

        # Each DT-generated rule represents one complete root-to-leaf path.
        # For one student, only one path should match because each DT split is either <= threshold or > threshold.
        # Therefore, the matching list should contain exactly one rule.
        selected_rule = matching_rules[0]

        # Store final ES output.
        self.result = {
            "prediction_code": selected_rule["prediction_code"],
            "prediction_label": selected_rule["prediction_label"],
            "rule_id": selected_rule["rule_id"],
            "confidence": selected_rule["confidence"],
            "explanation": selected_rule["conditions_text"]
        }


## ES: Define Prediction Function

Run ES inference for one processed student.


In [19]:
def infer_processed_student(processed_student_row, rules=knowledge_base):
    """Run ES inference for one processed test row."""

    # Create a fresh ES engine so this prediction is independent.
    engine = AcademicRiskExpertSystem(rules)

    # Reset the engine to make sure it starts with no previous facts.
    engine.reset()

    # Convert the processed student row from a pandas Series into a dictionary.
    student_features = processed_student_row.to_dict()

    # Create an Experta Student fact from the processed feature values.
    student_fact = Student(**student_features)

    # Add the Student fact to the ES engine.
    engine.declare(student_fact)

    # Run the ES inference process.
    engine.run()

    # Return the prediction produced by the matching rule.
    return engine.result

## LR: Train Model

Train Logistic Regression using the shared processed training data.


In [20]:
# LR is trained directly on the same processed feature matrix used by ES and SVM.
# class_weight="balanced" helps handle target class imbalance.
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

# Fit LR on the processed training set.
lr_model.fit(X_train_processed, y_train)

print("LR trained.")


LR trained.


## SVM: Train Model

Train SVM using the shared processed training data.


In [21]:
# SVM is trained directly on the same processed feature matrix used by ES and LR.
# kernel="rbf" allows a nonlinear decision boundary.
# class_weight="balanced" helps handle target class imbalance.
svm_model = SVC(
    kernel="rbf",
    class_weight="balanced",
    random_state=RANDOM_STATE
)

# Fit SVM on the processed training set.
svm_model.fit(X_train_processed, y_train)

print("SVM trained.")


SVM trained.


## ES: Generate Predictions

Generate ES predictions on the processed test set.


In [22]:
# Run ES inference for every processed test row.
es_results = [
    infer_processed_student(row)
    for _, row in X_test_processed.iterrows()
]

# Extract numeric prediction codes from ES results.
es_predictions = np.array([
    result["prediction_code"]
    for result in es_results
])


## LR and SVM: Generate Predictions

Generate LR and SVM predictions on the same processed test set.


In [23]:
# Predict test labels using LR.
lr_predictions = lr_model.predict(X_test_processed)

# Predict test labels using SVM.
svm_predictions = svm_model.predict(X_test_processed)

# Store all predictions in one dictionary for shared evaluation.
predictions = {
    "ES": es_predictions,
    "LR": lr_predictions,
    "SVM": svm_predictions
}


## Common: Evaluation Metrics

Calculate test-set metrics for ES, LR, and SVM.


In [24]:
def calculate_metrics(y_true, y_pred):
    """Calculate classification metrics."""

    # Accuracy: overall percentage of correct predictions.
    # Precision: among predicted At Risk cases, how many were actually At Risk.
    # Recall: among actual At Risk cases, how many were found.
    # F1-score: harmonic mean of precision and recall.
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }


# Build one comparison row per system.
metrics_table = pd.DataFrame([
    {"System": system_name, **calculate_metrics(y_test, y_pred)}
    for system_name, y_pred in predictions.items()
])

# Round metrics for cleaner display.
for column in ["Accuracy", "Precision", "Recall", "F1-score"]:
    metrics_table[column] = metrics_table[column].round(4)

display(metrics_table)


,System,Accuracy,Precision,Recall,F1-score
0,ES,0.8608,0.7586,0.8462,0.8000
1,LR,0.9114,0.8519,0.8846,0.8679
2,SVM,0.8861,0.7576,0.9615,0.8475


## Common: Detailed Reports

Print confusion matrices and classification reports for all systems.


In [25]:
def build_confusion_matrix_table(y_true, y_pred):
    """Return a readable confusion matrix table."""

    # Labels are ordered as:
    # 0 = Not At Risk
    # 1 = At Risk
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return pd.DataFrame(
        cm,
        index=["Actual: Not At Risk", "Actual: At Risk"],
        columns=["Predicted: Not At Risk", "Predicted: At Risk"]
    )


def build_classification_metrics(y_true, y_pred):
    """Return binary classification metrics for the positive class."""

    return {
        # Overall percentage of correct predictions.
        "Accuracy": round(
            accuracy_score(y_true, y_pred),
            4
        ),

        # Precision for class 1.
        "Precision": round(
            precision_score(
                y_true,
                y_pred,
                average="binary",
                pos_label=1,
                zero_division=0
            ),
            4
        ),

        # Recall for class 1.
        "Recall": round(
            recall_score(
                y_true,
                y_pred,
                average="binary",
                pos_label=1,
                zero_division=0
            ),
            4
        ),

        # F1-score for class 1.
        # This is binary F1, not macro, micro, or weighted F1.
        "F1-score": round(
            f1_score(
                y_true,
                y_pred,
                average="binary",
                pos_label=1,
                zero_division=0
            ),
            4
        )
    }


for system_name, y_pred in predictions.items():
    print(f"\n{system_name} - Positive Class: At Risk")

    metrics_table = pd.DataFrame(
        [build_classification_metrics(y_test, y_pred)]
    )

    display(metrics_table)

    confusion_matrix_table = build_confusion_matrix_table(y_test, y_pred)

    display(confusion_matrix_table)


ES - Positive Class: At Risk


,Accuracy,Precision,Recall,F1-score
0,0.8608,0.7586,0.8462,0.8


,Predicted: Not At Risk,Predicted: At Risk
Actual: Not At Risk,46,7
Actual: At Risk,4,22



LR - Positive Class: At Risk


,Accuracy,Precision,Recall,F1-score
0,0.9114,0.8519,0.8846,0.8679


,Predicted: Not At Risk,Predicted: At Risk
Actual: Not At Risk,49,4
Actual: At Risk,3,23



SVM - Positive Class: At Risk


,Accuracy,Precision,Recall,F1-score
0,0.8861,0.7576,0.9615,0.8475


,Predicted: Not At Risk,Predicted: At Risk
Actual: Not At Risk,45,8
Actual: At Risk,1,25


## Summary

The notebook compares:

- **ES:** Experta expert system using DT-generated rules
- **LR:** Logistic Regression
- **SVM:** Support Vector Machine

All systems use the same preprocessing, the same train/test split, and the same test-set metrics.
